# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.27.1 — FAST
## Nested Cubic Multiscale — Minimal Numerical Benchmark

Toy benchmark only.

We test a hierarchy
\[
\mathcal C_0\subset \mathcal C_1\subset\cdots\subset\mathcal C_{K-1}
\]
where each scale owns an orthonormal cubic triad.

Goals:
1. verify triad orthonormality at every scale;
2. verify composition of relative rotations;
3. show a cubic directional witness changes with scale;
4. test whether explicit multiscale averaging can reduce finite-scale anisotropy.

No physical coarse-graining law, 3D→4D emergence law, or new GVH dynamics is claimed.

In [1]:
import math, json
from pathlib import Path
import numpy as np
import pandas as pd

UPSTREAM = {
    "p3327_source_sha256":"2222a9127583cb6cefb9fdea82b8fb55505b6469a7fda30220468988504c65ca",
    "G27_KINEMATIC_GATE_REFERENCE_PASS":True,
    "G27_GVH_DISTINCT_EMERGENCE_LAW_DERIVED":False,
    "G27_NEW_GVH_PHYSICS_VALIDATED":False,
}
UPSTREAM_SCOPE_GATE = (
    UPSTREAM["G27_KINEMATIC_GATE_REFERENCE_PASS"]
    and not UPSTREAM["G27_GVH_DISTINCT_EMERGENCE_LAW_DERIVED"]
    and not UPSTREAM["G27_NEW_GVH_PHYSICS_VALIDATED"]
)
assert UPSTREAM_SCOPE_GATE
print("UPSTREAM_SCOPE_GATE =",UPSTREAM_SCOPE_GATE)
print("P3327_SOURCE_SHA256 =",UPSTREAM["p3327_source_sha256"])

UPSTREAM_SCOPE_GATE = True
P3327_SOURCE_SHA256 = 2222a9127583cb6cefb9fdea82b8fb55505b6469a7fda30220468988504c65ca


## Cubic frames

For an orthonormal frame \(E_k\), define
\[
Q_k(\mathbf n)=\sum_{A=1}^3(e^{(k)}_{(A)}\cdot \mathbf n)^4,\qquad |\mathbf n|=1.
\]

For one cubic frame:
\[
\frac13\le Q_k\le 1.
\]

For a uniform orientation average:
\[
\langle Q\rangle=\frac35.
\]

In [2]:
def Rx(a):
    c,s=np.cos(a),np.sin(a)
    return np.array([[1,0,0],[0,c,-s],[0,s,c]],float)
def Ry(a):
    c,s=np.cos(a),np.sin(a)
    return np.array([[c,0,s],[0,1,0],[-s,0,c]],float)
def Rz(a):
    c,s=np.cos(a),np.sin(a)
    return np.array([[c,-s,0],[s,c,0],[0,0,1]],float)
def euler_R(ax,ay,az):
    return Rz(az) @ Ry(ay) @ Rx(ax)
def Qc(E,n):
    p=E @ n
    return float(np.sum(p**4))

angles_deg=[
    (0,0,0),(17,31,11),(-23,14,37),(41,-19,26),
    (-11,47,-33),(29,22,-41),(-37,-28,19),(13,-44,52),
]
relative_R=[euler_R(*(np.deg2rad(x) for x in a)) for a in angles_deg]
K=len(relative_R)

frames=[np.eye(3)]
for k in range(1,K):
    frames.append(relative_R[k] @ frames[-1])

orth_errors=[np.max(np.abs(E@E.T-np.eye(3))) for E in frames]
NESTED_CUBES_ORTHONORMAL_PASS=max(orth_errors)<1e-14
assert NESTED_CUBES_ORTHONORMAL_PASS

print("K_levels =",K)
print("max_orthonormality_error =",max(orth_errors))
print("NESTED_CUBES_ORTHONORMAL_PASS =",NESTED_CUBES_ORTHONORMAL_PASS)

K_levels = 8
max_orthonormality_error = 6.661338147750939e-16
NESTED_CUBES_ORTHONORMAL_PASS = True


## Composition across scales

Require
\[
R_{m\leftarrow0}=R_{m\leftarrow m-1}\cdots R_{1\leftarrow0}.
\]

In [3]:
composition_errors=[]
for m in range(1,K):
    direct=np.eye(3)
    for j in range(1,m+1):
        direct=relative_R[j] @ direct
    composition_errors.append(np.max(np.abs(direct-frames[m])))

NESTED_ROTATION_COMPOSITION_PASS=max(composition_errors)<1e-14
assert NESTED_ROTATION_COMPOSITION_PASS

print("max_composition_error =",max(composition_errors))
print("NESTED_ROTATION_COMPOSITION_PASS =",NESTED_ROTATION_COMPOSITION_PASS)

max_composition_error = 0.0
NESTED_ROTATION_COMPOSITION_PASS = True


## Same physical direction, different cubic scale

Fix
\[
\mathbf n_*=(1,0,0).
\]

Every level evaluates its own \(Q_k(\mathbf n_*)\).

In [4]:
n_star=np.array([1.0,0.0,0.0])
q_levels=[Qc(E,n_star) for E in frames]

level_table=pd.DataFrame({"level":np.arange(K),"Q_k(n*)":q_levels})
print(level_table.to_string(index=False))

MULTISCALE_CUBIC_WITNESS_VARIES_PASS=(max(q_levels)-min(q_levels))>0.1
assert MULTISCALE_CUBIC_WITNESS_VARIES_PASS
print("Q_range =",max(q_levels)-min(q_levels))
print("MULTISCALE_CUBIC_WITNESS_VARIES_PASS =",MULTISCALE_CUBIC_WITNESS_VARIES_PASS)

 level  Q_k(n*)
     0 1.000000
     1 0.572324
     2 0.408487
     3 0.866174
     4 0.473732
     5 0.750188
     6 0.574680
     7 0.513678
Q_range = 0.5915130021082791
MULTISCALE_CUBIC_WITNESS_VARIES_PASS = True


## Multiscale anisotropy score

For the first \(N\) levels:
\[
\bar Q_N(\mathbf n)=\frac1N\sum_{k=0}^{N-1}Q_k(\mathbf n).
\]

Define
\[
A_N=\frac{\sigma_{\mathbf n}(\bar Q_N)}
{\langle\bar Q_N\rangle_{\mathbf n}}.
\]

A lower \(A_N\) means a more isotropic effective witness under the chosen averaging rule.

In [5]:
def fibonacci_sphere(n=5000):
    i=np.arange(n,dtype=float)
    z=1.0-2.0*(i+0.5)/n
    phi=np.pi*(3.0-np.sqrt(5.0))*i
    r=np.sqrt(np.maximum(0.0,1.0-z*z))
    return np.column_stack([r*np.cos(phi),r*np.sin(phi),z])

dirs=fibonacci_sphere(5000)
Qmat=np.empty((K,len(dirs)))
for k,E in enumerate(frames):
    proj=dirs @ E.T
    Qmat[k]=np.sum(proj**4,axis=1)

rows=[]; anis=[]; means=[]
for N in range(1,K+1):
    qbar=np.mean(Qmat[:N],axis=0)
    mean=float(np.mean(qbar))
    std=float(np.std(qbar))
    A=std/mean
    means.append(mean); anis.append(A)
    rows.append([N,mean,std,A,abs(mean-0.6)])

anis_table=pd.DataFrame(rows,columns=[
    "N_levels","mean_Qbar","std_Qbar","anisotropy_A","abs(mean-3/5)"
])
print(anis_table.to_string(index=False))

SINGLE_CUBE_MEAN_PASS=abs(means[0]-0.6)<5e-5
MULTISCALE_ANISOTROPY_REDUCED_PASS=anis[-1]<anis[0]
ISOTROPIC_MEAN_STABLE_PASS=max(abs(m-0.6) for m in means)<1e-4

assert SINGLE_CUBE_MEAN_PASS
assert MULTISCALE_ANISOTROPY_REDUCED_PASS
assert ISOTROPIC_MEAN_STABLE_PASS

print("A_1 =",anis[0])
print("A_K =",anis[-1])
print("anisotropy_reduction_factor =",anis[-1]/anis[0])
print("MULTISCALE_ANISOTROPY_REDUCED_PASS =",MULTISCALE_ANISOTROPY_REDUCED_PASS)

 N_levels  mean_Qbar  std_Qbar  anisotropy_A  abs(mean-3/5)
        1   0.600000  0.174574      0.290957   2.667214e-08
        2   0.599999  0.132846      0.221410   6.154005e-07
        3   0.599999  0.082055      0.136758   6.101284e-07
        4   0.599999  0.086217      0.143695   6.514046e-07
        5   0.600000  0.060925      0.101541   2.508370e-07
        6   0.600000  0.072388      0.120647   4.806047e-07
        7   0.600000  0.056860      0.094767   4.994995e-07
        8   0.600000  0.046770      0.077949   4.552185e-07
A_1 = 0.2909570980595187
A_K = 0.07794931602088606
anisotropy_reduction_factor = 0.267906562653923
MULTISCALE_ANISOTROPY_REDUCED_PASS = True


## Weight sensitivity

Equal weighting is not a physical law.

Compare
\[
w_k=\frac1K
\]
with a deliberately local-dominated weighting. If the result changes, the weights must eventually be derived rather than chosen.

In [6]:
w_equal=np.ones(K)/K
w_local=np.array([0.50,0.20,0.10,0.07,0.05,0.04,0.025,0.015],float)
w_local=w_local/w_local.sum()

def effective_anisotropy(weights):
    qeff=np.tensordot(weights,Qmat,axes=(0,0))
    return float(np.std(qeff)/np.mean(qeff))

A_equal=effective_anisotropy(w_equal)
A_local=effective_anisotropy(w_local)

WEIGHT_CHOICE_MATTERS_PASS=abs(A_equal-A_local)>1e-3
assert WEIGHT_CHOICE_MATTERS_PASS

print("A_equal =",A_equal)
print("A_local_dominated =",A_local)
print("WEIGHT_CHOICE_MATTERS_PASS =",WEIGHT_CHOICE_MATTERS_PASS)

A_equal = 0.07794931602088605
A_local_dominated = 0.16645139280926052
WEIGHT_CHOICE_MATTERS_PASS = True


## Verdict

This benchmark can establish only:
- nested cubic bookkeeping;
- consistent scale-rotation composition;
- scale-dependent cubic orientation information;
- possible anisotropy suppression under an explicit toy average.

It cannot establish a physical GVH multiscale law.

In [7]:
G271_NESTED_CUBIC_HIERARCHY_MATERIALIZED=True
G271_ROTATION_COMPOSITION_PASS=NESTED_ROTATION_COMPOSITION_PASS
G271_CUBIC_WITNESS_VARIES_BY_SCALE_PASS=MULTISCALE_CUBIC_WITNESS_VARIES_PASS
G271_EQUAL_WEIGHT_ANISOTROPY_REDUCTION_PASS=MULTISCALE_ANISOTROPY_REDUCED_PASS
G271_WEIGHT_CHOICE_MATTERS_PASS=WEIGHT_CHOICE_MATTERS_PASS

G271_MINIMAL_NUMERICAL_GATE_PASS=all([
    UPSTREAM_SCOPE_GATE,
    NESTED_CUBES_ORTHONORMAL_PASS,
    G271_ROTATION_COMPOSITION_PASS,
    G271_CUBIC_WITNESS_VARIES_BY_SCALE_PASS,
    G271_EQUAL_WEIGHT_ANISOTROPY_REDUCTION_PASS,
    G271_WEIGHT_CHOICE_MATTERS_PASS,
])

G271_PHYSICAL_COARSE_GRAINING_LAW_DERIVED=False
G271_3D_TO_4D_MULTISCALE_EMERGENCE_DERIVED=False
G271_LORENTZ_RECOVERY_FROM_MULTISCALE_CUBES_PROVED=False
G271_CORE_CHANGED=False
G271_NEW_GVH_PHYSICS_VALIDATED=False

G271_NEXT_AUTHORIZED=(
    "DERIVE-CANDIDATE-INTERSCALE-MAP-AND-TEST-COMPOSITION-ISOTROPIC-FIXED-POINT"
    if G271_MINIMAL_NUMERICAL_GATE_PASS
    else "REPAIR-NESTED-CUBIC-MINIMAL-NUMERICAL-BENCHMARK"
)

assert G271_MINIMAL_NUMERICAL_GATE_PASS
assert not G271_PHYSICAL_COARSE_GRAINING_LAW_DERIVED
assert not G271_3D_TO_4D_MULTISCALE_EMERGENCE_DERIVED
assert not G271_NEW_GVH_PHYSICS_VALIDATED

for name in [
    "G271_NESTED_CUBIC_HIERARCHY_MATERIALIZED",
    "G271_ROTATION_COMPOSITION_PASS",
    "G271_CUBIC_WITNESS_VARIES_BY_SCALE_PASS",
    "G271_EQUAL_WEIGHT_ANISOTROPY_REDUCTION_PASS",
    "G271_WEIGHT_CHOICE_MATTERS_PASS",
    "G271_MINIMAL_NUMERICAL_GATE_PASS",
    "G271_PHYSICAL_COARSE_GRAINING_LAW_DERIVED",
    "G271_3D_TO_4D_MULTISCALE_EMERGENCE_DERIVED",
    "G271_CORE_CHANGED",
    "G271_NEW_GVH_PHYSICS_VALIDATED",
]:
    print(name,"=",globals()[name])
print("G271_NEXT_AUTHORIZED =",G271_NEXT_AUTHORIZED)

G271_NESTED_CUBIC_HIERARCHY_MATERIALIZED = True
G271_ROTATION_COMPOSITION_PASS = True
G271_CUBIC_WITNESS_VARIES_BY_SCALE_PASS = True
G271_EQUAL_WEIGHT_ANISOTROPY_REDUCTION_PASS = True
G271_WEIGHT_CHOICE_MATTERS_PASS = True
G271_MINIMAL_NUMERICAL_GATE_PASS = True
G271_PHYSICAL_COARSE_GRAINING_LAW_DERIVED = False
G271_3D_TO_4D_MULTISCALE_EMERGENCE_DERIVED = False
G271_CORE_CHANGED = False
G271_NEW_GVH_PHYSICS_VALIDATED = False
G271_NEXT_AUTHORIZED = DERIVE-CANDIDATE-INTERSCALE-MAP-AND-TEST-COMPOSITION-ISOTROPIC-FIXED-POINT


In [8]:
artifact={
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.27.1_Nested_Cubic_Multiscale_Minimal_Numerical_Benchmark_FAST",
    "classification":"TOY_MULTISCALE_BOOKKEEPING_BENCHMARK",
    "levels":K,
    "relative_euler_angles_deg":angles_deg,
    "single_direction_Q_levels":[float(x) for x in q_levels],
    "anisotropy_table":anis_table.to_dict(orient="records"),
    "effective_anisotropy":{"equal":A_equal,"local_dominated":A_local},
    "flags":{
        "G271_MINIMAL_NUMERICAL_GATE_PASS":G271_MINIMAL_NUMERICAL_GATE_PASS,
        "G271_PHYSICAL_COARSE_GRAINING_LAW_DERIVED":G271_PHYSICAL_COARSE_GRAINING_LAW_DERIVED,
        "G271_3D_TO_4D_MULTISCALE_EMERGENCE_DERIVED":G271_3D_TO_4D_MULTISCALE_EMERGENCE_DERIVED,
        "G271_NEW_GVH_PHYSICS_VALIDATED":G271_NEW_GVH_PHYSICS_VALIDATED,
        "G271_NEXT_AUTHORIZED":G271_NEXT_AUTHORIZED,
    },
    "scope_note":"Equal-weight averaging is illustrative only; no physical coarse-graining law is derived."
}

export_dir=Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/"gvh_0.3.2.7.3.7.3.3.27.1_Nested_Cubic_Multiscale_Minimal_Numerical_Benchmark_FAST.json"
artifact_path.write_text(json.dumps(artifact,indent=2,ensure_ascii=False,default=lambda o:o.item() if isinstance(o,np.generic) else str(o)),encoding="utf-8")
print("artifact =",artifact_path)

artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.27.1_Nested_Cubic_Multiscale_Minimal_Numerical_Benchmark_FAST.json


### Provenance

`AI-F + C + TOY-MULTISCALE-BENCHMARK`

Not `NOV-PASS`.